## Building 3-km public ES-ILRC/SNED/SPED pairs

In [19]:
# Check OSRM connection
import requests
try:
    resp = requests.get("http://localhost:5000/route/v1/driving/121.0,14.5;121.1,14.5", timeout=10)
    data = resp.json()
    if data.get("code") == "Ok":
        dist_km = data["routes"][0]["distance"] / 1000
        print("OSRM server is running and reachable.")
        print(f"end=to-end distance: {dist_km:.2f} km")
    else:
        print(f"OSRM server returned an error: {data.get('code')}")
except requests.RequestException as e:
    print(f"Error occurred while fetching OSRM data: {e}")

OSRM server is running and reachable.
end=to-end distance: 14.59 km


In [20]:
from pathlib import Path

# Threshold
ACCESS_THRESHOLD_M        = 3_000    # the 3 km coverage threshold for this analysis
HAVERSINE_BUFFER_M        = 9_000    # 3x threshold — safety margin for road tortuosity in the pre-filter
SNAP_OFFSHORE_THRESHOLD_M = 5_000    # flags GPS points that don't snap to a real road (same as before)
OSRM_BATCH_SIZE           = 200
CHECKPOINT_EVERY          = 250      # facility roster is small (~2.9k), so checkpoint more often
EARTH_R                   = 6_371_000
OSRM_BASE = 'http://localhost:5000'

# ── Paths — point ROOT at your new project directory ─────────────────────────
ROOT         = Path.cwd().parent
DATA         = ROOT / 'data'
INTERMEDIATE_DIR      = DATA / 'intermediate'
EXTERNAL_DIR = DATA / 'external'
OUT_DIR      = DATA

ILRC_2223_CSV    = INTERMEDIATE_DIR / 'ILRCS_2223.csv'
ILRC_DLFUNDS_CSV = INTERMEDIATE_DIR / 'ILRCS_DL_FUNDS.csv'
SNED_CSV         = INTERMEDIATE_DIR / 'SNED_SCHOOLS.csv'
SPED_CENTER_CSV  = INTERMEDIATE_DIR / 'SPED_CENTERS.csv'

PUB_PARQUET = EXTERNAL_DIR / 'public_school_coordinates.parquet'
BRGY_GPKG   = EXTERNAL_DIR / 'consolidated_geodata_matched.gpkg'

CHECKPOINT_PATH = OUT_DIR / 'checkpoint_sned_pairs.parquet'

### 1. Load four rosters and build unified facility list

In [21]:
import pandas as pd

PSGC_COLS = ['psgc_region', 'psgc_province', 'psgc_municity', 'psgc_barangay']
PSGC_DTYPE = {'school_id': str, **{c: str for c in PSGC_COLS}}

ilrc_2223    = pd.read_csv(ILRC_2223_CSV,    dtype=PSGC_DTYPE)
ilrc_dlfunds = pd.read_csv(ILRC_DLFUNDS_CSV, dtype=PSGC_DTYPE)
sned         = pd.read_csv(SNED_CSV,         dtype=PSGC_DTYPE)
sped_center  = pd.read_csv(SPED_CENTER_CSV,  dtype=PSGC_DTYPE)

for name, df in [('ILRC 22-23', ilrc_2223), ('ILRC DL Funds', ilrc_dlfunds),
                  ('SNED', sned), ('SPED Centers', sped_center)]:
    print(f'{name:16s}: {len(df):5,} rows | {df["school_id"].nunique():5,} unique IDs | '
          f'coord_status: {df["coord_status"].value_counts(dropna=False).to_dict()}')


def _clean(df: pd.DataFrame, flag_col: str) -> pd.DataFrame:
    """Drop exact-duplicate rows and rows with no school_id (unusable — can't
    be geolocated or deduped against the other three lists), then tag the
    designation flag for this source."""
    df = df.copy()
    before = len(df)
    df = df.drop_duplicates()
    if len(df) < before:
        print(f'  [{flag_col}] dropped {before - len(df)} exact-duplicate row(s)')

    no_id = df['school_id'].isna()
    if no_id.any():
        print(f'  [{flag_col}] dropping {no_id.sum()} row(s) with no school_id (manual follow-up needed):')
        print(df.loc[no_id, ['school_name']].to_string(index=False))
    df = df[~no_id].copy()

    df[flag_col] = True
    return df

ilrc_2223_c    = _clean(ilrc_2223,    'is_ilrc')
ilrc_dlfunds_c = _clean(ilrc_dlfunds, 'is_ilrc')
sned_c         = _clean(sned,         'is_sned')
sped_center_c  = _clean(sped_center,  'is_sped_center')

# keep track of which ILRC funding tranche a school came from, in case it's useful later
ilrc_2223_c['ilrc_source']    = '2223'
ilrc_dlfunds_c['ilrc_source'] = 'dl_funds'

facility_raw = pd.concat([ilrc_2223_c, ilrc_dlfunds_c, sned_c, sped_center_c], ignore_index=True)
print(f'\nConcatenated (pre-dedup): {len(facility_raw):,} rows')

# ── collapse to one row per unique school_id ─────────────────────────────────
# A school can appear in more than one list (e.g. an SNED school that is also a
# SPED Center). We do NOT want two separate points at the same coordinates —
# that would double-count coverage and waste OSRM calls. Instead we OR the
# designation flags together and keep one location record per school.
FLAG_COLS = ['is_ilrc', 'is_sned', 'is_sped_center']
for c in FLAG_COLS:
    facility_raw[c] = facility_raw.get(c, False)
    facility_raw[c] = facility_raw[c].fillna(False)

LOCATION_COLS = [
    'school_name', 'latitude', 'longitude', 'coord_status', 'coord_source',
    'psgc_region', 'psgc_region_name', 'psgc_province', 'psgc_province_name',
    'psgc_municity', 'psgc_municity_name', 'psgc_barangay', 'psgc_barangay_name',
    'offers_es', 'offers_jhs', 'offers_shs', 'school_management', 'enrollment_status',
]
# Location/PSGC fields are identical across source files for the same school_id
# (they all trace back to the same underlying DepEd gold roster), so 'first' is safe.
# NOTE: descriptive fields unique to one source (infra_status, mdt_status, district,
# sector, etc.) are NOT carried into this deduped table — if you need the full
# per-source detail (e.g. "why is this school an ILRC candidate"), keep facility_raw
# itself as a lookup table alongside this one.

agg = {c: 'any' for c in FLAG_COLS}
agg.update({c: 'first' for c in LOCATION_COLS})
facility = facility_raw.groupby('school_id', as_index=False).agg(agg)

n_multi = (facility[FLAG_COLS].sum(axis=1) > 1).sum()
print(f'\nUnique special-ed facility schools: {len(facility):,}')
print(f'  is_ilrc:        {facility["is_ilrc"].sum():,}')
print(f'  is_sned:        {facility["is_sned"].sum():,}')
print(f'  is_sped_center: {facility["is_sped_center"].sum():,}')
print(f'  Multi-designation schools: {n_multi:,}')

facility.to_parquet(OUT_DIR / 'sned_facility_roster.parquet', index=False)
facility.to_csv(OUT_DIR / 'sned_facility_roster.csv', index=False)
print(f'Saved → {OUT_DIR / "sned_facility_roster.parquet"}')

ILRC 22-23      :    32 rows |    32 unique IDs | coord_status: {'valid': 32}
ILRC DL Funds   :    28 rows |    25 unique IDs | coord_status: {'valid': 28}
SNED            : 2,392 rows | 2,392 unique IDs | coord_status: {'valid': 2376, 'suspect': 15, 'no_coords': 1}
SPED Centers    :   471 rows |   471 unique IDs | coord_status: {'valid': 464, 'suspect': 7}
  [is_ilrc] dropped 2 exact-duplicate row(s)

Concatenated (pre-dedup): 2,921 rows

Unique special-ed facility schools: 2,629
  is_ilrc:        57
  is_sned:        2,392
  is_sped_center: 471
  Multi-designation schools: 274
Saved → c:\Users\Lanz Railey Fermin\Desktop\ecair_work\sned-analysis\data\sned_facility_roster.parquet


### 2. Proxy coords

In [22]:
import geopandas as gpd
import requests

brgy = gpd.read_file(BRGY_GPKG)
rep = brgy.geometry.representative_point()   # guaranteed inside the polygon, unlike .centroid
brgy['centroid_lat'] = rep.y
brgy['centroid_lng'] = rep.x
brgy_lookup = brgy.set_index('psgc_code')[['centroid_lat', 'centroid_lng']].to_dict('index')
print(f'Barangay polygons loaded: {len(brgy_lookup):,}')

def _get_proxy(row, brgy_lookup):
    status = row['coord_status']
    if status in ('valid', 'suspect', 'fixed_swap') and pd.notna(row['latitude']):
        tier = 'spatial' if status == 'valid' else 'road_snap_suspect'
        return row['latitude'], row['longitude'], tier
    if status == 'no_coords':
        code = str(row['psgc_barangay']).strip() if pd.notna(row['psgc_barangay']) else None
        if code and code in brgy_lookup:
            c = brgy_lookup[code]
            return c['centroid_lat'], c['centroid_lng'], 'road_snap_barangay'
    return None, None, 'unroutable'

proxies = facility.apply(_get_proxy, axis=1, args=(brgy_lookup,), result_type='expand')
proxies.columns = ['proxy_lat', 'proxy_lng', 'proxy_tier']
facility = pd.concat([facility, proxies], axis=1)
print(facility['proxy_tier'].value_counts())

# Offshore/bad-GPS check for the 'suspect' rows (same Fix-C logic as the old pipeline):
# if the coordinate doesn't snap to a real road within 5 km, it's unusable for routing.
def _check_offshore(lat, lng):
    try:
        r = requests.get(f'{OSRM_BASE}/nearest/v1/driving/{lng},{lat}', timeout=5)
        data = r.json()
        if data.get('code') == 'Ok' and data.get('waypoints'):
            return data['waypoints'][0]['distance'] <= SNAP_OFFSHORE_THRESHOLD_M
    except Exception:
        pass
    return False

suspect_idx = facility.index[facility['proxy_tier'] == 'road_snap_suspect']
n_demoted = 0
for idx in suspect_idx:
    lat, lng = facility.at[idx, 'proxy_lat'], facility.at[idx, 'proxy_lng']
    if not _check_offshore(lat, lng):
        facility.at[idx, ['proxy_lat', 'proxy_lng']] = None
        facility.at[idx, 'proxy_tier'] = 'unroutable'
        n_demoted += 1
print(f'Offshore check: {len(suspect_idx)} suspect facilities checked, {n_demoted} demoted to unroutable')

facility_routable = facility[facility['proxy_tier'] != 'unroutable'].reset_index(drop=True)
print(f'Routable facilities: {len(facility_routable):,} / {len(facility):,}')

Barangay polygons loaded: 42,048
proxy_tier
spatial              2608
road_snap_suspect      20
unroutable              1
Name: count, dtype: int64
Offshore check: 20 suspect facilities checked, 0 demoted to unroutable
Routable facilities: 2,628 / 2,629


### 3. Load public school universe

In [23]:
pub = pd.read_parquet(PUB_PARQUET)
pub = pub[pub['coord_status'].isin(['valid'])].copy()  # add the suspect on-land check from the old
                                                     # loaders.py if you want the exact same rigor
for col in ['offers_es', 'offers_jhs', 'offers_shs']:
    pub[col] = pub[col].map({'True': True, 'False': False}).fillna(False)
pub['school_id'] = pub['school_id'].astype(str)
pub = pub.reset_index(drop=True)
print(f'Population (public schools, valid coords): {len(pub):,}')

Population (public schools, valid coords): 46,537


In [24]:
# ── To extend the population to private schools too, uncomment: ─────────────
priv = pd.read_parquet(EXTERNAL_DIR / 'private_school_coordinates.parquet')
priv = priv[priv['coord_status'].isin(['valid'])].copy()
priv['school_id'] = priv['school_id'].astype(str)
pub = pd.concat([pub, priv], ignore_index=True)

print(f'Population (priv schools): {len(priv):,}')
print(f'Population (pub+priv schools, valid coords): {len(pub):,}')

Population (priv schools): 7,582
Population (pub+priv schools, valid coords): 54,119


In [25]:
8825-7582

1243

### 4. BallTree

In [26]:
import numpy as np
from sklearn.neighbors import BallTree

pub_rad = np.radians(pub[['latitude', 'longitude']].values.astype(np.float64))
tree = BallTree(pub_rad, metric='haversine')
print(f'BallTree built on {len(pub):,} population schools.')

BallTree built on 54,119 population schools.


### 5. OSRM routing loop

In [27]:
import time

def _haversine_km(lat1, lng1, lat2, lng2):
    r1, r2 = np.radians([lat1, lng1]), np.radians([lat2, lng2])
    dlat, dlng = r2[0] - r1[0], r2[1] - r1[1]
    a = np.sin(dlat/2)**2 + np.cos(r1[0])*np.cos(r2[0])*np.sin(dlng/2)**2
    return 2 * EARTH_R * np.arcsin(np.sqrt(a)) / 1000

def _osrm_distances(src_lat, src_lng, dest_coords):
    coords = [f'{src_lng},{src_lat}'] + [f'{lng},{lat}' for lat, lng in dest_coords]
    coord_str = ';'.join(coords)
    dest_idx = ';'.join(str(i) for i in range(1, len(coords)))
    url = (f'{OSRM_BASE}/table/v1/driving/{coord_str}'
           f'?sources=0&destinations={dest_idx}&annotations=distance')
    try:
        r = requests.get(url, timeout=30)
        data = r.json()
        if data.get('code') == 'Ok':
            return data['distances'][0]
    except Exception:
        pass
    return [None] * len(dest_coords)

def _osrm_batched(src_lat, src_lng, dest_latlng):
    out = []
    for i in range(0, len(dest_latlng), OSRM_BATCH_SIZE):
        out.extend(_osrm_distances(src_lat, src_lng, dest_latlng[i:i + OSRM_BATCH_SIZE]))
    return out

threshold_rad = HAVERSINE_BUFFER_M / EARTH_R

processed_ids = set()
pair_rows = []
if CHECKPOINT_PATH.exists():
    ckpt = pd.read_parquet(CHECKPOINT_PATH)
    pair_rows = ckpt.to_dict('records')
    processed_ids = set(ckpt['facility_school_id'].unique())
    print(f'Resuming: {len(processed_ids):,} facilities already processed')
else:
    print('No checkpoint found — starting fresh.')

to_process = facility_routable[~facility_routable['school_id'].isin(processed_ids)]
print(f'Facilities to route: {len(to_process):,} (of {len(facility_routable):,} routable)', flush=True)

t0 = time.time()
n_done = 0
for _, row in to_process.iterrows():
    src_lat, src_lng = row['proxy_lat'], row['proxy_lng']
    idx = tree.query_radius(np.radians([[src_lat, src_lng]]), r=threshold_rad)[0]

    if len(idx) > 0:
        cand = pub.iloc[idx]
        dest_latlng = list(zip(cand['latitude'], cand['longitude']))
        road_m = _osrm_batched(src_lat, src_lng, dest_latlng)

        for pop_row, dist_m in zip(cand.itertuples(), road_m):
            if dist_m is not None and dist_m <= ACCESS_THRESHOLD_M:
                hav_km = _haversine_km(src_lat, src_lng, pop_row.latitude, pop_row.longitude)
                pair_rows.append({
                    'facility_school_id':      row['school_id'],
                    'facility_school_name':    row['school_name'],
                    'facility_is_ilrc':        bool(row['is_ilrc']),
                    'facility_is_sned':        bool(row['is_sned']),
                    'facility_is_sped_center': bool(row['is_sped_center']),
                    'facility_proxy_tier':     row['proxy_tier'],
                    'population_school_id':          str(pop_row.school_id),
                    'population_school_name':        pop_row.school_name,
                    'population_psgc_municity':      str(pop_row.psgc_municity) if pd.notna(pop_row.psgc_municity) else None,
                    'population_psgc_municity_name': pop_row.psgc_municity_name,
                    'road_distance_km':       round(dist_m / 1000, 3),
                    'haversine_distance_km':  round(hav_km, 3),
                })

    processed_ids.add(row['school_id'])
    n_done += 1

    if n_done % CHECKPOINT_EVERY == 0:
        pd.DataFrame(pair_rows).to_parquet(CHECKPOINT_PATH, index=False)
        elapsed = time.time() - t0
        rate = n_done / elapsed
        remaining = (len(to_process) - n_done) / rate if rate > 0 else float('inf')
        print(f'  [{n_done:,}/{len(to_process):,}] {len(pair_rows):,} pairs | '
              f'{elapsed/60:.1f} min elapsed | ~{remaining/60:.1f} min remaining', flush=True)

if pair_rows:
    pd.DataFrame(pair_rows).to_parquet(CHECKPOINT_PATH, index=False)
print(f'Routing complete. Pairs within {ACCESS_THRESHOLD_M/1000:.0f} km: {len(pair_rows):,}')

No checkpoint found — starting fresh.
Facilities to route: 2,628 (of 2,628 routable)
  [250/2,628] 2,501 pairs | 0.4 min elapsed | ~3.7 min remaining
  [500/2,628] 6,299 pairs | 0.9 min elapsed | ~3.7 min remaining
  [750/2,628] 8,697 pairs | 1.2 min elapsed | ~2.9 min remaining
  [1,000/2,628] 11,226 pairs | 1.4 min elapsed | ~2.2 min remaining
  [1,250/2,628] 13,461 pairs | 1.6 min elapsed | ~1.8 min remaining
  [1,500/2,628] 14,347 pairs | 1.8 min elapsed | ~1.3 min remaining
  [1,750/2,628] 16,366 pairs | 2.1 min elapsed | ~1.0 min remaining
  [2,000/2,628] 18,658 pairs | 2.4 min elapsed | ~0.7 min remaining
  [2,250/2,628] 21,451 pairs | 2.8 min elapsed | ~0.5 min remaining
  [2,500/2,628] 27,109 pairs | 3.6 min elapsed | ~0.2 min remaining
Routing complete. Pairs within 3 km: 28,438


### 6. Save pairs and relevant reports

In [33]:
pairs_df = pd.DataFrame(pair_rows)
for col in ['road_distance_km', 'haversine_distance_km']:
    pairs_df[col] = pairs_df[col].astype('float32')

pairs_df.to_parquet(OUT_DIR / 'sned_access_pubpriv_pairs.parquet', index=False)
pairs_df.to_csv(OUT_DIR / 'sned_access_pubpriv_pairs.csv', index=False)
print(f'Saved {len(pairs_df):,} rows → sned_access_pubpriv_pairs.parquet / .csv')

Saved 28,438 rows → sned_access_pubpriv_pairs.parquet / .csv


In [34]:
cov = pairs_df.groupby('population_school_id').agg(
    n_facilities_within_3km=('facility_school_id', 'count'),
    n_ilrc_within_3km=('facility_is_ilrc', 'sum'),
    n_sned_within_3km=('facility_is_sned', 'sum'),
    n_sped_center_within_3km=('facility_is_sped_center', 'sum'),
    nearest_facility_km=('road_distance_km', 'min'),
).reset_index()

summary = pub[['school_id', 'school_name', 'psgc_region_name', 'psgc_province_name',
               'psgc_municity', 'psgc_municity_name']].rename(columns={'school_id': 'population_school_id'})

summary = summary.merge(cov, on='population_school_id', how='left')
count_cols = ['n_facilities_within_3km', 'n_ilrc_within_3km', 'n_sned_within_3km', 'n_sped_center_within_3km']
summary[count_cols] = summary[count_cols].fillna(0).astype(int)
summary['has_sned_access_3km'] = summary['n_facilities_within_3km'] > 0
# nearest_facility_km stays NaN where there's no facility within 3 km — that's informative, leave it

summary.to_parquet(OUT_DIR / 'sned_access_pubpriv_summary.parquet', index=False)
summary.to_csv(OUT_DIR / 'sned_access_pubpriv_summary.csv', index=False)
print(f'Saved {len(summary):,} rows → sned_access_pubpriv_summary.parquet / .csv')

n_gap = (~summary['has_sned_access_3km']).sum()
print(f'Public schools WITHOUT ILRC/SPED Center/SNED access within 3 km: '
      f'{n_gap:,} / {len(summary):,} ({n_gap/len(summary):.1%})')

Saved 54,119 rows → sned_access_pubpriv_summary.parquet / .csv
Public schools WITHOUT ILRC/SPED Center/SNED access within 3 km: 37,182 / 54,119 (68.7%)


In [35]:
lgu = summary.groupby('psgc_municity').agg(
    psgc_municity_name=('psgc_municity_name', 'first'),
    psgc_province_name=('psgc_province_name', 'first'),
    psgc_region_name=('psgc_region_name', 'first'),
    n_public_schools=('population_school_id', 'count'),
    n_with_access=('has_sned_access_3km', 'sum'),
).reset_index()
lgu['n_without_access'] = lgu['n_public_schools'] - lgu['n_with_access']
lgu['pct_without_access'] = (lgu['n_without_access'] / lgu['n_public_schools']).round(3)

# Strongest signal: municipalities that have NO ILRC/SPED Center/SNED school at all,
# anywhere within their borders (not just "not within 3 km of a given school").
facility_lgus = set(facility_routable['psgc_municity'].dropna().astype(str))
lgu['lgu_has_no_facility_at_all'] = ~lgu['psgc_municity'].astype(str).isin(facility_lgus)

lgu = lgu.sort_values('n_without_access', ascending=False)
lgu.to_parquet(OUT_DIR / 'sned_access_pubpriv_lgu_counts.parquet', index=False)
lgu.to_csv(OUT_DIR / 'sned_access_pubpriv_lgu_counts.csv', index=False)
print(f'Saved {len(lgu):,} rows → sned_access_pubpriv_lgu_counts.parquet / .csv')

Saved 1,635 rows → sned_access_pubpriv_lgu_counts.parquet / .csv


In [36]:
print('\n' + '=' * 60)
print('SNED ACCESS (ILRC / SPED Center / SNED) — SUMMARY')
print('=' * 60)
print(f'Facility roster (unique schools): {len(facility):,}')
print(f'  is_ilrc:        {facility["is_ilrc"].sum():,}')
print(f'  is_sned:        {facility["is_sned"].sum():,}')
print(f'  is_sped_center: {facility["is_sped_center"].sum():,}')
print(f'  Routable:       {len(facility_routable):,}')
print()
print(f'Population public schools checked: {len(summary):,}')
print(f'  WITH access within 3 km:    {summary["has_sned_access_3km"].sum():,}')
print(f'  WITHOUT access within 3 km: {n_gap:,} ({n_gap/len(summary):.1%})')
print()
print(f'Municipalities with zero facility at all: {lgu["lgu_has_no_facility_at_all"].sum():,} / {len(lgu):,}')
print('=' * 60)


SNED ACCESS (ILRC / SPED Center / SNED) — SUMMARY
Facility roster (unique schools): 2,629
  is_ilrc:        57
  is_sned:        2,392
  is_sped_center: 471
  Routable:       2,628

Population public schools checked: 54,119
  WITH access within 3 km:    16,937
  WITHOUT access within 3 km: 37,182 (68.7%)

Municipalities with zero facility at all: 543 / 1,635


In [37]:
on_disk = pd.read_csv(OUT_DIR / 'sned_access_pubpriv_pairs.csv')
assert len(on_disk) == len(pairs_df), (
    f"Stale save mismatch: wrote {len(pairs_df):,} rows but disk has {len(on_disk):,}"
)